# Baseline Model – DistilBERT (Single-label smoke test)

Goal:
- lightweight transformer baseline
- first working training pipeline
- temporary single-label setup (for debugging/training sanity check)

In [18]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

In [19]:
dataset = load_dataset("go_emotions", "simplified")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [20]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [21]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

print(tokenized_dataset)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5427
    })
})


In [22]:
def is_single_label(example):
    return len(example["labels"]) == 1

single_label_dataset = tokenized_dataset.filter(is_single_label)

print(single_label_dataset)

Filter:   0%|          | 0/43410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 36308
    })
    validation: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4548
    })
    test: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4590
    })
})


In [23]:
def collapse_label(example):
    example["labels"] = example["labels"][0]
    return example

single_label_dataset = single_label_dataset.map(collapse_label)

print(single_label_dataset["train"][0]["labels"])
print(type(single_label_dataset["train"][0]["labels"]))

Map:   0%|          | 0/36308 [00:00<?, ? examples/s]

Map:   0%|          | 0/4548 [00:00<?, ? examples/s]

Map:   0%|          | 0/4590 [00:00<?, ? examples/s]

27
<class 'int'>


In [24]:
single_label_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

sample = single_label_dataset["train"][0]
print(sample.keys())
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"])

dict_keys(['labels', 'input_ids', 'attention_mask'])
torch.Size([128])
torch.Size([128])
tensor(27)


In [25]:
num_labels = dataset["train"].features["labels"].feature.num_classes
print("num_labels =", num_labels)

num_labels = 28


In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("Model num_labels:", model.config.num_labels)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model num_labels: 28


In [27]:
train_size = min(1000, len(single_label_dataset["train"]))
val_size = min(200, len(single_label_dataset["validation"]))

small_train = single_label_dataset["train"].select(range(train_size))
small_val = single_label_dataset["validation"].select(range(val_size))

print("Train size:", len(small_train))
print("Val size:", len(small_val))

Train size: 1000
Val size: 200


In [28]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [29]:
training_args = TrainingArguments(
    output_dir="./baseline_distilbert_results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=100,
    report_to="none",
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics,
)

print(trainer)

In [31]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=2.704587906125992, metrics={'train_runtime': 756.1441, 'train_samples_per_second': 1.322, 'train_steps_per_second': 0.083, 'total_flos': 33132205056000.0, 'train_loss': 2.704587906125992, 'epoch': 1.0})

In [32]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 2.7056884765625,
 'eval_accuracy': 0.325,
 'eval_macro_f1': 0.018867924528301886,
 'eval_runtime': 41.1699,
 'eval_samples_per_second': 4.858,
 'eval_steps_per_second': 0.316,
 'epoch': 1.0}